# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring.** This notebook writes the contract for the slice I will
use, checks every claim with a query on the real warehouse, builds five features, and springs the leakage
trap once on purpose.

**Decision moment:** `2026-03-31` (end of a mid-panel month). **Features:** March. **Label:** April — a
forward window, not the sealed final month (June). The token is read from the environment or the Colab
Secret `HF_TOKEN`; it is never written in a cell.

Everything below runs against `hf://datasets/FlyRank/internship-warehouse` through DuckDB, cached locally
by `huggingface_hub` so the queries are fast.

## 1. The contract in plain words (5 answers)

**1) What one row means for my lane.** At warehouse grain, one row = one **`report_date × client_hash_id ×
content_hash_id`** — a page's performance on a single day. For the queue it collapses to one row per page
over a 31-day window; both grains are stated because the query below checks the first one.

**2) Which tables.** `fact_content_daily_performance` (March for features, April for the label),
`dim_content` (content age and type), `dim_clients` (client count and history coverage).

**3) Which time window.** Features: **2026-03-01 → 2026-03-31**. Decision moment: **2026-03-31 23:59**.
Label: **2026-04-01 → 2026-04-30**. Minimum volume: **100 March impressions** (the lane guide's minimum-
volume idea, not a universal truth).

**4) What I would predict or rank.** Rank pages by the probability that **April impressions fall below 80%
of March impressions** — a forward decline proxy. The product is a ranked queue of 50 pages with an action
and reason codes; this notebook only builds the first five features toward it.

**5) One thing I deliberately exclude.** Every row where **`ga4_data_available IS NOT TRUE`**. Those GA4
columns are zero-filled when the flag is not true, so treating them as "no engagement" would turn "no
tracking" into a fake signal. Excluding them makes this contract **search-only**, which is the honest
scope of the release for most rows.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

try:
    import duckdb
    from huggingface_hub import hf_hub_download
except ImportError:                     # Colab without them: install quietly
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"])
    import duckdb
    from huggingface_hub import hf_hub_download

# Token order: env var (local) -> Colab Secret. Never paste a token into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face READ token: ")

REPO = "FlyRank/internship-warehouse"
FILES = {
    "march": "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "april": "fact_content_daily_performance/month=2026-04/data_0.parquet",
    "dim_content": "dim_content.parquet",
    "dim_clients": "dim_clients.parquet",
}
local = {name: hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)
         for name, path in FILES.items()}

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
for name, path in local.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{Path(path).as_posix()}')")

CLIENTS_TOTAL = con.sql("SELECT COUNT(*) FROM dim_clients").fetchall()[0][0]
print("warehouse slices ready (cached locally):")
for name, path in local.items():
    print(f"  {name:<12}{Path(path).stat().st_size / 1e6:>7.1f} MB")
print(f"clients in dim_clients: {CLIENTS_TOTAL}")


warehouse slices ready (cached locally):
  march         124.2 MB
  april         134.0 MB
  dim_content    19.6 MB
  dim_clients     0.0 MB
clients in dim_clients: 104


## 2. Fields: feature / label / context / excluded

Every field I touch goes in exactly one bucket. The trap column is already listed here — as a **label
source**, never a feature — because the field list is written before the model sees it.

| Field | Bucket | Why |
|---|---|---|
| `report_date`, `client_hash_id`, `content_hash_id` | context | keys for grouping, splitting and joining — never model inputs |
| `impressions_30d` | feature | trailing March sum, closed at the decision moment |
| `ctr_30d` | feature | clicks ÷ impressions over the same window |
| `avg_position_30d` | feature | average GSC position over the same window |
| `days_with_impressions_30d` | feature | persistence: how many March days had ≥ 1 impression |
| `content_age_days` | feature | `2026-03-31 − content_created_date` from `dim_content` |
| `declined_next_30d` | label | April outcome — never a feature |
| `impressions_next_30d` | label source | the April sum the label is computed from; excluded from features (this is the trap) |
| `ga4_*` columns | excluded | zero-filled when `ga4_data_available` is not true |
| `last_optimized_date` | excluded | 91% NULL (474,210 of 519,606) — a missingness signal, not a freshness one |
| `is_published`, `is_deleted` | excluded | product decisions, not observed performance |
| `content_type` | context | used for reading and slicing; not one of the five features |

In [2]:
buckets = pd.DataFrame([
    ("report_date, client_hash_id, content_hash_id", "context", "keys for grouping / splitting / joining"),
    ("impressions_30d", "feature", "trailing March sum, closed at the decision moment"),
    ("ctr_30d", "feature", "clicks / impressions x 100, same window"),
    ("avg_position_30d", "feature", "average GSC position, same window"),
    ("days_with_impressions_30d", "feature", "how many March days had an impression"),
    ("content_age_days", "feature", "2026-03-31 minus content_created_date"),
    ("declined_next_30d", "label", "April outcome, never a feature"),
    ("impressions_next_30d", "label source", "April sum the label is built from — the trap"),
    ("ga4_*", "excluded", "zero-filled when ga4_data_available is not true"),
    ("last_optimized_date", "excluded", "91% NULL, a missingness signal not freshness"),
    ("is_published, is_deleted", "excluded", "product decisions, not observed performance"),
], columns=["field", "bucket", "why"])
for row in buckets.itertuples(index=False):
    print(f"  {row[0]:<46}{row[1]:<14}{row[2]}")

  report_date, client_hash_id, content_hash_id  context       keys for grouping / splitting / joining
  impressions_30d                               feature       trailing March sum, closed at the decision moment
  ctr_30d                                       feature       clicks / impressions x 100, same window
  avg_position_30d                              feature       average GSC position, same window
  days_with_impressions_30d                     feature       how many March days had an impression
  content_age_days                              feature       2026-03-31 minus content_created_date
  declined_next_30d                             label         April outcome, never a feature
  impressions_next_30d                          label source  April sum the label is built from — the trap
  ga4_*                                         excluded      zero-filled when ga4_data_available is not true
  last_optimized_date                           excluded      91% NULL, a miss

## 3. Verify it with queries (grain, counts, availability)

Exactly three verification queries — one per claim that would otherwise be a guess. The feature build
after them is not a verification query; it is the work the contract enables.

### 3a — Query 1: the grain

*Claim: one row = one date × client × content. Expectation: **zero** groups with more than one row.*

In [3]:
duplicates = con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_key
FROM march
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

print(f"duplicate (date, client, content) groups: {len(duplicates)}")
print("-> grain holds: zero duplicate rows." if duplicates.empty else duplicates)

duplicate (date, client, content) groups: 0
-> grain holds: zero duplicate rows.


### 3b — Query 2: how big is the slice, and over which dates

*Claim: March is one calendar month of daily facts, roughly 9–10M rows. Expectation: 31 days, many clients
and pages, no date outside March.*

In [4]:
span = con.sql("""
SELECT COUNT(*)                            AS rows,
       MIN(report_date)                    AS first_day,
       MAX(report_date)                    AS last_day,
       DATEDIFF('day', MIN(report_date), MAX(report_date)) + 1 AS days,
       COUNT(DISTINCT client_hash_id)      AS clients,
       COUNT(DISTINCT content_hash_id)     AS pages
FROM march
""").df()
span

,rows,first_day,last_day,days,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


### 3c — Query 3: availability, checked with `IS TRUE`

*Claim: the fact table has rows for page-days that carry no search or analytics data. The filter that
matters is `IS TRUE`, not `= 1` and not `IS NOT FALSE` — NULL is a third state here. Expectation: a large
share of rows does **not** survive the GSC filter.*

In [5]:
avail = con.sql("""
SELECT COUNT(*)                                           AS rows_total,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS NULL) AS ga4_unknown
FROM march
""").df()
avail["gsc_share"] = (avail["gsc_rows"] / avail["rows_total"]).round(3)
avail["ga4_share"] = (avail["ga4_rows"] / avail["rows_total"]).round(3)
avail["ga4_unknown_share"] = (avail["ga4_unknown"] / avail["rows_total"]).round(3)

print(f"rows surviving gsc_data_available IS TRUE: {int(avail.loc[0, 'gsc_rows']):,}"
      f" of {int(avail.loc[0, 'rows_total']):,} = {avail.loc[0, 'gsc_share']:.1%}")
print(f"rows surviving ga4_data_available IS TRUE: {int(avail.loc[0, 'ga4_rows']):,}"
      f" = {avail.loc[0, 'ga4_share']:.1%} of the slice")
print(f"ga4_data_available NULL (neither true nor false): "
      f"{int(avail.loc[0, 'ga4_unknown']):,} = {avail.loc[0, 'ga4_unknown_share']:.1%}")
print("-> engagement features would keep only the GA4-true rows; this contract stays search-only.")

rows surviving gsc_data_available IS TRUE: 3,611,061 of 9,841,378 = 36.7%
rows surviving ga4_data_available IS TRUE: 413,966 = 4.2% of the slice
ga4_data_available NULL (neither true nor false): 3,018,741 = 30.7%
-> engagement features would keep only the GA4-true rows; this contract stays search-only.


## 3d. Five features, each knowable at the decision moment

Max five, and every one answers the same question: *could a reviewer standing at 2026-03-31 23:59 know
this value without looking at April?*

| # | Feature | Definition | Knowable at the decision moment because… |
|---|---|---|---|
| 1 | `impressions_30d` | SUM of `gsc_impressions`, Mar 1–31 | the window closed at the decision moment |
| 2 | `ctr_30d` | `gsc_clicks ÷ gsc_impressions × 100`, same window | clicks and impressions in the same closed window |
| 3 | `avg_position_30d` | AVG `gsc_avg_position` over days with position > 0 | GSC records position for days already elapsed |
| 4 | `days_with_impressions_30d` | count of March days with ≥ 1 impression | a day-by-day count inside the closed window |
| 5 | `content_age_days` | `2026-03-31 − content_created_date` | creation date is fixed in `dim_content` before the decision |

The label is built in the same cell, from April only. Pages with **no April row** are dropped, not
labelled "declined" — an absent row can mean deleted, untracked, or a client that stopped reporting, and
the data cannot tell those apart.

In [6]:
frame = con.sql("""
WITH m AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_30d,
           SUM(gsc_clicks)      AS clicks_30d,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_30d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d
    FROM march
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
),
a AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_next_30d
    FROM april
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
)
SELECT m.client_hash_id, m.content_hash_id,
       m.impressions_30d,
       m.clicks_30d * 100.0 / m.impressions_30d AS ctr_30d,
       m.avg_position_30d,
       m.days_with_impressions_30d,
       DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
       a.impressions_next_30d,
       c.content_type,
       (a.impressions_next_30d < 0.8 * m.impressions_30d) AS declined_next_30d
FROM m
LEFT JOIN a USING (client_hash_id, content_hash_id)
JOIN dim_content c USING (client_hash_id, content_hash_id)
""").df()

dropped = int(frame["impressions_next_30d"].isna().sum())
feat = (frame.dropna(subset=["impressions_next_30d"])
             .sort_values(["client_hash_id", "content_hash_id"])
             .reset_index(drop=True))
feat["declined_next_30d"] = feat["declined_next_30d"].astype(int)

print(f"pages with >=100 March impressions: {len(frame):,}")
print(f"pages with no April row (dropped, not labelled decline): {dropped}")
print(f"feature frame: {len(feat):,} pages | {feat['client_hash_id'].nunique()} clients"
      f" | base rate {feat['declined_next_30d'].mean():.3f}")
print(f"missing avg_position: {int(feat['avg_position_30d'].isna().sum()):,} (kept; the model fills 0)")
feat.drop(columns=["client_hash_id", "content_hash_id"]).head(5)

pages with >=100 March impressions: 101,441
pages with no April row (dropped, not labelled decline): 548
feature frame: 100,893 pages | 43 clients | base rate 0.515
missing avg_position: 0 (kept; the model fills 0)


,impressions_30d,ctr_30d,avg_position_30d,days_with_impressions_30d,content_age_days,impressions_next_30d,content_type,declined_next_30d
0,331.0,0.60423,14.129210,31,175,561.0,keyword article,0
1,145.0,0.00000,8.470926,30,175,23.0,keyword article,1
2,461.0,0.00000,14.859827,31,175,964.0,keyword article,0
3,232.0,0.00000,12.297523,29,175,248.0,keyword article,0
4,464.0,0.00000,9.554793,18,175,1138.0,keyword article,0


### 3e — The trap, sprung once on purpose

`impressions_next_30d` is the April column the label is computed from. A model that uses it is not
predicting the future — it is reading the answer key. I add it as a sixth feature, watch the quick score
jump toward perfect, **delete it**, and keep the honest number.

Same split for all three scores: 20% of **clients** held out, so no client leaks across the boundary.

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

FEATURES = ["impressions_30d", "ctr_30d", "avg_position_30d",
            "days_with_impressions_30d", "content_age_days"]

X = feat[FEATURES].fillna(0)
y = feat["declined_next_30d"].values
groups = feat["client_hash_id"].values
train, test = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
                   .split(X, y, groups))

def quick_auc(columns):
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=1000, class_weight="balanced"))
    model.fit(columns.iloc[train], y[train])
    return roc_auc_score(y[test], model.predict_proba(columns.iloc[test])[:, 1])

honest = quick_auc(X)

X_leak = X.assign(impressions_next_30d=feat["impressions_next_30d"])
leaky = quick_auc(X_leak)
del X_leak                                  # the leak column is deleted, not kept "for later"
honest_again = quick_auc(X)

print(f"held-out clients: {feat['client_hash_id'].iloc[test].nunique()} | pages: {len(test):,}")
print(f"  honest 5-feature score              AUC {honest:.3f}")
print(f"  + impressions_next_30d (the leak)   AUC {leaky:.3f}")
print(f"  after deleting the leak             AUC {honest_again:.3f}")
print("\nthe jump is not skill: the label is a threshold on that exact column,")
print("so the model is reading the answer key. Delete it and keep the honest number.")

held-out clients: 9 | pages: 15,191
  honest 5-feature score              AUC 0.591
  + impressions_next_30d (the leak)   AUC 0.998
  after deleting the leak             AUC 0.591

the jump is not skill: the label is a threshold on that exact column,
so the model is reading the answer key. Delete it and keep the honest number.


## 4. One named limitation of this slice

**The slice is small, unbalanced, and search-only.** The contract is honest about it with numbers: the
feature frame covers fewer clients than the warehouse, GSC is present on roughly a third of March page-days,
and GA4 on about one in twenty — with another 30% of GA4 flags NULL rather than false. Any engagement
feature would silently drop almost the whole slice, which is why this contract stops at search signals.

In [8]:
print(f"feature frame: {len(feat):,} pages across {feat['client_hash_id'].nunique()} of {CLIENTS_TOTAL} clients")
print(f"GSC available on {avail.loc[0, 'gsc_share']:.1%} of March rows"
      f" | GA4 on {avail.loc[0, 'ga4_share']:.1%} | GA4 flag NULL on {avail.loc[0, 'ga4_unknown_share']:.1%}")
print(f"pages dropped for having no April row: {dropped}")
print(f"-> claims describe {feat['client_hash_id'].nunique()} clients with >=100 March impressions,")
print(f"   not the {CLIENTS_TOTAL}-client warehouse, and they say nothing about engagement.")

feature frame: 100,893 pages across 43 of 104 clients
GSC available on 36.7% of March rows | GA4 on 4.2% | GA4 flag NULL on 30.7%
pages dropped for having no April row: 548
-> claims describe 43 clients with >=100 March impressions,
   not the 104-client warehouse, and they say nothing about engagement.


## Self-check

- [x] Five plain-words contract answers, in §1
- [x] Exactly three verification queries, outputs visible — grain, counts/span, availability with `IS TRUE`
- [x] Five features, each with a "knowable at the decision moment because…" line
- [x] The deliberate leak is shown, removed, and the honest score kept
- [x] One named limitation with its numbers
- [x] No client names, URLs, raw queries, or tokens anywhere — only pseudonymous hashes
- [x] Runs top to bottom (`Runtime → Run all`), then committed under `work/notebooks/`